In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


# Import e caricamento automatico dei file della competizione.
import json
import subprocess
from collections import Counter
from pathlib import Path

from tqdm.auto import tqdm
import xgboost as xgb


gpu_check = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader",
    ],
    capture_output=True,
    text=True,
)

if gpu_check.returncode != 0:
    raise RuntimeError(
        "GPU NVIDIA non disponibile. In Kaggle apri Settings > Accelerator e seleziona GPU."
    )

print("GPU:", gpu_check.stdout.strip())
print("XGBoost:", xgb.__version__)

xgb_major_version = int(
    xgb.__version__.split(".", 1)[0]
)

if xgb_major_version >= 2:
    XGB_GPU_PARAMS = {
        "tree_method": "hist",
        "device": "cuda",
        "fail_on_invalid_gpu_id": True,
    }
else:
    # Compatibilit? con immagini Kaggle pi? vecchie.
    XGB_GPU_PARAMS = {
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
    }


INPUT_ROOT = Path("/kaggle/input")
required_files = {
    "train.jsonl",
    "test.jsonl",
    "type_chart_gen2.csv",
}

DATA_DIR = next(
    (
        path.parent
        for path in INPUT_ROOT.rglob("train.jsonl")
        if required_files.issubset({
            child.name
            for child in path.parent.iterdir()
            if child.is_file()
        })
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Non trovo train.jsonl, test.jsonl e type_chart_gen2.csv nella stessa cartella sotto /kaggle/input."
    )


def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
        ]


train_data = load_jsonl(
    DATA_DIR / "train.jsonl"
)
test_data = load_jsonl(
    DATA_DIR / "test.jsonl"
)
type_chart = pd.read_csv(
    DATA_DIR / "type_chart_gen2.csv"
)

print("Cartella dati:", DATA_DIR)
print(
    f"Train: {len(train_data):,} | Test: {len(test_data):,}"
)
print("Type chart:", type_chart.shape)


/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/sample_submission.csv
/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/test.jsonl
/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/train.jsonl
/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/type_chart_gen2.csv
GPU: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
XGBoost: 3.4.1
Cartella dati: /kaggle/input/competitions/fds-pokemon-battles-prediction-2026
Train: 10,000 | Test: 5,000
Type chart: (110, 3)


# FDS Pokemon Battles Prediction 2026 - Kaggle GPU

Pipeline `Run All` con un solo XGBoost su GPU: stato compatto, conteggi delle mosse, duello finale, andamento temporale e matchup della squadra rimanente.

L'ablation e il tuning usano una sola GPU in modo sequenziale; la submission viene salvata in `/kaggle/working/`.


## 1. Funzioni comuni per ricostruire lo stato della battaglia


In [2]:
# Costanti e funzioni condivise dalle feature finali.


STATS = ('base_hp', 'base_atk', 'base_def', 'base_spa', 'base_spd', 'base_spe')


BOOSTS = ('atk', 'def', 'spa', 'spd', 'spe')


STATUSES = ('nostatus', 'brn', 'frz', 'par', 'psn', 'slp', 'tox')


CATEGORIES = ('physical', 'special', 'status')


SIDES = ('p1', 'p2')


WINDOWS = {'all': (1, 30), '01_10': (1, 10), '11_20': (11, 20), '21_30': (21, 30), 'last5': (26, 30)}


PLACEHOLDERS = {'', 'missing', 'none', 'null', 'nomove', 'no_move', 'noaction', 'no_action', 'switch'}


def normalize_effects(value):
    """None = dato ignoto; noeffect/lista vuota = assenza osservata."""
    if value is None:
        return None
    if isinstance(value, str):
        value = [value]
    return frozenset(clean(v) for v in value
                     if clean(v) not in {'noeffect', 'none', 'missing', ''})


def clean(value):
    return str(value if value is not None else 'missing').strip().lower().replace(' ', '_')


def number(value):
    if value is None:
        return None
    try:
        value = float(value)
    except (TypeError, ValueError):
        return None
    return value if np.isfinite(value) else None


def valid_hp(value):
    value = number(value)
    return value if value is not None and 0 <= value <= 1 else None


def build_pokedex(battles):
    """Solo attributi intrinseci: niente strumenti, target o statistiche d'uso."""
    result = {}
    for battle in battles:
        rows = list(battle.get('p1_team_details') or [])
        if battle.get('p2_lead_details'):
            rows.append(battle['p2_lead_details'])
        for row in rows:
            if row.get('name'):
                attributes = {key: row[key] for key in (*STATS, 'types') if key in row}
                result.setdefault(clean(row['name']), {}).update(attributes)
    return result


POKEDEX = build_pokedex(train_data)
TYPE_MULTIPLIER = {
    (clean(row.attacking_type), clean(row.defending_type)): float(row.multiplier)
    for row in type_chart.itertuples(index=False)
}
KNOWN_TYPES = {pokemon_type for pair in TYPE_MULTIPLIER for pokemon_type in pair}


def observed_timeline(battle):
    events = [e for e in (battle.get('battle_timeline') or [])
              if 1 <= int(e.get('turn', 0)) <= 30]
    events.sort(key=lambda e: int(e['turn']))
    turns = [int(e['turn']) for e in events]
    if len(turns) != len(set(turns)):
        raise ValueError('Attesa una sola osservazione per turno nella battle_timeline.')
    return events


def metadata_for(battle):
    metadata = dict(POKEDEX)
    rows = list(battle.get('p1_team_details') or [])
    if battle.get('p2_lead_details'):
        rows.append(battle['p2_lead_details'])
    for row in rows:
        if row.get('name'):
            name = clean(row['name'])
            metadata[name] = {**metadata.get(name, {}), **row}
    return metadata


def types_of(row):
    return [clean(t) for t in (row.get('types') or [])
            if clean(t) not in {'notype', 'none', 'missing', ''}]


def type_effectiveness(attack_type, defender_types):
    """Nel CSV sparso le coppie non elencate valgono 1, per tipi conosciuti."""
    attack_type = clean(attack_type)
    if attack_type not in KNOWN_TYPES or not defender_types:
        return None
    if any(t not in KNOWN_TYPES for t in defender_types):
        return None
    return float(np.prod([TYPE_MULTIPLIER.get((attack_type, t), 1.0)
                          for t in defender_types]))


def move_profile(move):
    move = move or {}
    name = clean(move.get('name'))
    category = clean(move.get('category'))
    if move.get('name') is not None and name in PLACEHOLDERS:
        return None
    if name in PLACEHOLDERS and category not in CATEGORIES:
        return None
    power = number(move.get('base_power'))
    power = power if power is not None and power >= 0 else None
    accuracy = number(move.get('accuracy'))
    # Il formato puo essere percentuale o frazione; True significa accuratezza certa.
    if accuracy is not None and accuracy > 1:
        accuracy /= 100
    if accuracy is not None and not 0 <= accuracy <= 1:
        accuracy = None
    return {'category': category, 'power': power, 'accuracy': accuracy,
            'type': clean(move.get('type')),
            'key': name if name not in PLACEHOLDERS else
                   (category, clean(move.get('type')), power)}


def empty_state():
    return {'hp': None, 'hp_turn': None, 'status': None, 'status_turn': None, 'ko': False,
            'seen_turn': None, 'boosts': {}, 'moves': {},
            'effects': None, 'effects_turn': None}


def collect_battle(battle):
    events = observed_timeline(battle)
    states = {side: {} for side in SIDES}
    for row in battle.get('p1_team_details') or []:
        if row.get('name'):
            states['p1'].setdefault(clean(row['name']), empty_state())
    # P2 lead e conosciuto, ma senza una misurazione non assumiamo HP o status.
    lead = battle.get('p2_lead_details') or {}
    if lead.get('name'):
        states['p2'].setdefault(clean(lead['name']), empty_state())
    active = {side: None for side in SIDES}
    records = {side: [] for side in SIDES}
    for event in events:
        turn = int(event['turn'])
        for side, opponent in (('p1', 'p2'), ('p2', 'p1')):
            raw = event.get(f'{side}_pokemon_state') or {}
            name = clean(raw['name']) if raw.get('name') else None
            record = {'turn': turn, 'new_ko': 0, 'switch': 0,
                      'hp_delta': None, 'hp_gap': None, 'move': None,
                      'active_hp': None, 'active_status': None,
                      'active_name': name, 'active_effects': None,
                      'defender': (event.get(f'{opponent}_pokemon_state') or {}).get('name')}
            if name:
                state = states[side].setdefault(name, empty_state())
                record['switch'] = int(active[side] not in (None, name))
                active[side] = name
                state['seen_turn'] = turn
                status = clean(raw['status']) if raw.get('status') else None
                hp = valid_hp(raw.get('hp_pct'))
                is_ko = status == 'fnt' or hp == 0
                record['new_ko'] = int(is_ko and not state['ko'])
                state['ko'] = state['ko'] or is_ko
                if state['ko']:
                    hp, status = 0.0, 'fnt'
                # Per le medie nel tempo registriamo la misura di questo turno,
                # senza ripetere valori vecchi quando manca l'osservazione.
                record['active_hp'] = hp
                record['active_status'] = status
                effects = normalize_effects(raw.get('effects'))
                record['active_effects'] = effects
                # Un campo assente resta ignoto, non trasciniamo effetti vecchi.
                state['effects'] = effects
                state['effects_turn'] = turn if effects is not None else None
                if hp is not None:
                    if state['hp'] is not None:
                        record['hp_delta'] = hp - state['hp']
                        record['hp_gap'] = turn - state['hp_turn']
                    state['hp'], state['hp_turn'] = hp, turn
                if status is not None:
                    state['status'] = status
                    state['status_turn'] = turn
                state['boosts'] = {boost: number((raw.get('boosts') or {}).get(boost))
                                   for boost in BOOSTS}
            move = move_profile(event.get(f'{side}_move_details'))
            record['move'] = move
            if move is not None and name:
                states[side][name]['moves'][move['key']] = move
            records[side].append(record)
    return {'states': states, 'active': active, 'records': records,
            'metadata': metadata_for(battle),
            'end_turn': int(events[-1]['turn']) if events else 0}


def summarize(f, prefix, values):
    values = [float(v) for v in values if v is not None and np.isfinite(v)]
    f[prefix + '_known_count'] = float(len(values))
    f[prefix + '_mean'] = float(np.mean(values)) if values else 0.0
    f[prefix + '_min'] = min(values, default=0.0)
    f[prefix + '_max'] = max(values, default=0.0)


def timeline_features(context):
    f = {}
    metadata = context['metadata']
    for side in SIDES:
        records = context['records'][side]
        for window, (start, stop) in WINDOWS.items():
            rows = [r for r in records if start <= r['turn'] <= stop]
            prefix = f'timeline_{side}_{window}'
            f[prefix + '_observed_turns'] = float(len(rows))
            f[prefix + '_new_ko'] = float(sum(r['new_ko'] for r in rows))
            f[prefix + '_switches'] = float(sum(r['switch'] for r in rows))
            # Le variazioni dopo un'assenza sono separate da quelle consecutive.
            for kind, predicate in (('hp', lambda gap: gap == 1),
                                    ('hp_gap', lambda gap: gap > 1)):
                deltas = [r['hp_delta'] for r in rows if r['hp_delta'] is not None
                          and predicate(r['hp_gap'])]
                f[f'{prefix}_{kind}_pairs'] = float(len(deltas))
                f[f'{prefix}_{kind}_loss'] = sum(max(-d, 0.0) for d in deltas)
                f[f'{prefix}_{kind}_recovery'] = sum(max(d, 0.0) for d in deltas)
                f[f'{prefix}_{kind}_net'] = sum(deltas)
            moves = [r['move'] for r in rows if r['move'] is not None]
            f[prefix + '_move_count'] = float(len(moves))
            for category in CATEGORIES:
                f[f'{prefix}_move_{category}_fraction'] = (
                    sum(m['category'] == category for m in moves) / len(moves) if moves else 0.0)
            f[prefix + '_move_category_unknown_fraction'] = (
                sum(m['category'] not in CATEGORIES for m in moves) / len(moves) if moves else 0.0)
            attacking = [m for m in moves if m['category'] in ('physical', 'special')]
            summarize(f, prefix + '_attack_power', [m['power'] for m in attacking
                                                   if m['power'] is not None and m['power'] > 0])
            f[prefix + '_attack_power_unknown_or_zero_fraction'] = (
                sum(m['power'] is None or m['power'] == 0 for m in attacking) / len(attacking)
                if attacking else 0.0)
            summarize(f, prefix + '_accuracy', [m['accuracy'] for m in moves])
            effectiveness = []
            for row in rows:
                move = row['move']
                if move is not None and move['category'] in ('physical', 'special'):
                    defender = metadata.get(clean(row['defender']), {})
                    value = type_effectiveness(move['type'], types_of(defender))
                    if value is not None:
                        effectiveness.append(value)
            f[prefix + '_attack_effectiveness_known_count'] = float(len(effectiveness))
            for label, predicate in (('super_effective', lambda v: v > 1),
                                     ('resisted', lambda v: 0 < v < 1),
                                     ('immune', lambda v: v == 0)):
                f[f'{prefix}_{label}_fraction'] = (
                    sum(predicate(v) for v in effectiveness) / len(effectiveness) if effectiveness else 0.0)
        for metric, key in (('ko', 'new_ko'), ('switch', 'switch')):
            turns = [r['turn'] for r in records if r[key]]
            f[f'timeline_{side}_has_{metric}'] = float(bool(turns))
            f[f'timeline_{side}_turns_since_last_{metric}'] = (
                float(context['end_turn'] - max(turns)) if turns else 30.0)
        f[f'timeline_{side}_hp_momentum'] = (
            f[f'timeline_{side}_21_30_hp_net'] - f[f'timeline_{side}_11_20_hp_net'])
    for window in WINDOWS:
        for metric in ('new_ko', 'hp_loss', 'hp_recovery', 'hp_net', 'switches'):
            f[f'timeline_{window}_{metric}_difference'] = (
                f[f'timeline_p1_{window}_{metric}'] - f[f'timeline_p2_{window}_{metric}'])
    f['timeline_hp_momentum_difference'] = f['timeline_p1_hp_momentum'] - f['timeline_p2_hp_momentum']
    return f


def occupancy_features(context):
    """HP medio degli attivi, persistenza degli status e potenza per turno.

    Ogni osservazione ha lo stesso peso. Il valore descrive il Pokemon attivo
    registrato, non gli HP dell'intera squadra. Gli status persistenti vengono
    contati a ogni osservazione; fnt resta separato dalle alterazioni di stato.
    La potenza e quella registrata: un turno senza mossa contribuisce zero;
    un dato sconosciuto non viene interpretato come un attacco mancato.
    """
    f = {}
    for side in SIDES:
        for window, (start, stop) in WINDOWS.items():
            rows = [r for r in context['records'][side] if start <= r['turn'] <= stop]
            count = len(rows)
            prefix = f'occupancy_{side}_{window}'
            hp = [r['active_hp'] for r in rows if r['active_hp'] is not None]
            f[prefix + '_active_hp_mean'] = float(np.mean(hp)) if hp else 0.0
            f[prefix + '_active_hp_known_fraction'] = len(hp) / count if count else 0.0
            known_status = sum(r['active_status'] in (*STATUSES, 'fnt') for r in rows)
            f[prefix + '_status_known_fraction'] = known_status / count if count else 0.0
            # Denominatore: tutte le osservazioni della finestra, anche quelle
            # con status mancante. La frazione nota permette di distinguerle.
            for status in (*STATUSES[1:], 'fnt'):
                f[f'{prefix}_status_{status}_fraction'] = (
                    sum(r['active_status'] == status for r in rows) / count if count else 0.0)
            powers = [r['move']['power'] for r in rows
                      if r['move'] is not None and r['move']['power'] is not None]
            f[prefix + '_recorded_base_power_per_turn'] = sum(powers) / count if count else 0.0
            f[prefix + '_move_power_known_fraction'] = len(powers) / count if count else 0.0
    for window in WINDOWS:
        for metric in ('active_hp_mean', 'recorded_base_power_per_turn',
                       *(f'status_{status}_fraction' for status in (*STATUSES[1:], 'fnt'))):
            f[f'occupancy_{window}_{metric}_difference'] = (
                f[f'occupancy_p1_{window}_{metric}'] - f[f'occupancy_p2_{window}_{metric}'])
    return f


def resource_features(f, context):
    for side in SIDES:
        states = context['states'][side]
        alive = {name: state for name, state in states.items() if not state['ko']}
        observed = [s for s in states.values() if s['seen_turn'] is not None]
        hp = [s['hp'] for s in alive.values() if s['hp'] is not None]
        fainted = sum(s['ko'] for s in states.values())
        f[f'{side}_remaining'] = float(6 - fainted)
        f[f'{side}_known_members'] = float(len(states))
        f[f'{side}_revealed_members'] = float(len(observed))
        f[f'{side}_unidentified_members'] = float(max(0, 6 - len(states)))
        # Nessuna ipotesi di HP=100% per membri mai osservati.
        f[f'{side}_alive_hp_unknown_count'] = float(max(0, 6 - fainted - len(hp)))
        f[f'{side}_alive_observed_hp_sum'] = float(sum(hp))
        summarize(f, f'{side}_alive_hp', hp)
        for threshold, label in ((.25, 'under25'), (.5, 'under50')):
            f[f'{side}_alive_{label}'] = float(sum(v < threshold for v in hp))
        f[f'{side}_alive_over75'] = float(sum(v > .75 for v in hp))
        for status in STATUSES:
            f[f'{side}_alive_status_{status}'] = float(sum(s['status'] == status for s in alive.values()))
        f[f'{side}_alive_status_unknown_count'] = float(
            6 - fainted - sum(s['status'] in STATUSES for s in alive.values()))
        ages = [context['end_turn'] - s['hp_turn'] for s in alive.values() if s['hp_turn'] is not None]
        summarize(f, f'{side}_hp_observation_age', ages)
        for stat in STATS:
            values = [number(context['metadata'].get(name, {}).get(stat)) for name in alive]
            summarize(f, f'{side}_alive_{stat}', [v for v in values if v is not None and v > 0])
    for metric in ('remaining', 'alive_observed_hp_sum', 'alive_hp_mean',
                   'alive_under25', 'alive_under50', 'alive_over75'):
        f[metric + '_difference'] = f['p1_' + metric] - f['p2_' + metric]
    for status in STATUSES:
        f[f'status_{status}_difference'] = f[f'p1_alive_status_{status}'] - f[f'p2_alive_status_{status}']


def active_features(f, context):
    for side in SIDES:
        name = context['active'][side]
        state = context['states'][side].get(name)
        metadata = context['metadata'].get(name, {})
        prefix = f'active_{side}'
        f[prefix + '_known'] = float(state is not None)
        f[prefix + '_hp_known'] = float(state is not None and state['hp'] is not None)
        f[prefix + '_hp'] = state['hp'] if state is not None and state['hp'] is not None else 0.0
        f[prefix + '_observation_age'] = float(context['end_turn'] - state['seen_turn']) if state else 30.0
        for attribute in ('hp', 'status'):
            turn = state[attribute + '_turn'] if state else None
            f[f'{prefix}_{attribute}_observation_age'] = float(context['end_turn'] - turn) if turn is not None else 30.0
        for status in (*STATUSES, 'fnt'):
            f[f'{prefix}_status_{status}'] = float(state is not None and state['status'] == status)
        f[prefix + '_status_known'] = float(state is not None and state['status'] in (*STATUSES, 'fnt'))
        for stat in STATS:
            value = number(metadata.get(stat))
            f[f'{prefix}_{stat}_known'] = float(value is not None and value > 0)
            f[f'{prefix}_{stat}'] = value if value is not None and value > 0 else 0.0
        for boost in BOOSTS:
            value = state['boosts'].get(boost) if state else None
            f[f'{prefix}_boost_{boost}_known'] = float(value is not None)
            f[f'{prefix}_boost_{boost}'] = value if value is not None else 0.0
    for metric in ('hp', *STATS, *(f'boost_{b}' for b in BOOSTS)):
        f[f'active_{metric}_difference'] = f[f'active_p1_{metric}'] - f[f'active_p2_{metric}']


def offensive_coverage_features(context):
    """Otto indici generici della copertura offensiva osservata al cutoff.

    Per ogni difensore j, R_j = max_{i,m}(HP_i * P_m * E_mj * A_i/D_j).
    La pressione media e sum_j(HP_j * R_j) / sum_j(HP_j), limitata ai
    difensori valutabili. La copertura peggiore e min_j(R_j).

    Considera solo Pokemon non osservati KO, con HP noti e positivi, e mosse
    fisiche/speciali osservate con potenza positiva. Attacco e difesa sono
    statistiche BASE della categoria corrispondente, non statistiche effettive.
    Non e un simulatore di danno: non applica boost, STAB, accuratezza, effetti
    o strumenti. Le mosse a potenza nulla/variabile non sono valutate da questo
    indicatore. Gli HP sono gli ultimi osservati, anche se non recenti.

    Una coppia ignota viene esclusa, non valutata come neutra o immune.
    Un'immunita conosciuta produce invece una risposta valida pari a zero.
    Le frazioni usano come denominatore 6 meno i KO osservati, includendo
    quindi anche i membri non identificati o non valutabili.
    Nessun nome entra nelle colonne e nessuna mossa viene inferita dal tipo.
    """
    if not TYPE_MULTIPLIER:
        raise ValueError('Caricare type_chart prima della cella delle feature, oppure chiamare '
                         'configure_reference_data(training_battles, type_chart).')
    f = {}
    states, metadata = context['states'], context['metadata']
    category_stats = {'physical': ('base_atk', 'base_def'),
                      'special': ('base_spa', 'base_spd')}
    for side in SIDES:
        opponent = 'p2' if side == 'p1' else 'p1'
        remaining_attackers = 6 - sum(s['ko'] for s in states[side].values())
        remaining_defenders = 6 - sum(s['ko'] for s in states[opponent].values())
        candidates = []
        eligible_attackers = set()
        for name, state in states[side].items():
            if state['ko'] or state['hp'] is None or state['hp'] <= 0:
                continue
            for move in state['moves'].values():
                if move['category'] not in category_stats:
                    continue
                if move['power'] is None or move['power'] <= 0 or move['type'] not in KNOWN_TYPES:
                    continue
                attack_stat, defense_stat = category_stats[move['category']]
                attack = number(metadata.get(name, {}).get(attack_stat))
                if attack is None or attack <= 0:
                    continue
                candidates.append((state['hp'], move, attack, defense_stat))
                eligible_attackers.add(name)

        responses, defender_hp = [], []
        for name, state in states[opponent].items():
            if state['ko'] or state['hp'] is None or state['hp'] <= 0:
                continue
            defender_metadata = metadata.get(name, {})
            defender_types = types_of(defender_metadata)
            scores = []
            for hp, move, attack, defense_stat in candidates:
                defense = number(defender_metadata.get(defense_stat))
                if defense is None or defense <= 0:
                    continue
                effect = type_effectiveness(move['type'], defender_types)
                if effect is None:
                    continue
                scores.append(hp * move['power'] * effect * attack / defense)
            if scores:
                responses.append(max(scores))
                defender_hp.append(state['hp'])

        prefix = f'coverage_{side}'
        hp_sum = sum(defender_hp)
        f[prefix + '_offensive_pressure'] = (
            float(sum(hp * response for hp, response in zip(defender_hp, responses)) / hp_sum)
            if hp_sum else 0.0)
        f[prefix + '_worst_response'] = float(min(responses)) if responses else 0.0
        f[prefix + '_attackers_evaluable_fraction'] = (
            len(eligible_attackers) / remaining_attackers if remaining_attackers > 0 else 0.0)
        f[prefix + '_defenders_evaluable_fraction'] = (
            len(responses) / remaining_defenders if remaining_defenders > 0 else 0.0)
    return f


def _potential_alive(context, side):
    return [(name, state) for name, state in context['states'][side].items()
            if not state['ko'] and state['hp'] is not None and state['hp'] > 0]


def _potential_remaining(context, side):
    return max(0, 6 - sum(state['ko'] for state in context['states'][side].values()))


def _potential_pair_pressure(context, attacker_name, attacker, defender_name, defender):
    """Indice, NON danno percentuale: max(P * acc * STAB * E * A/D)/(HP% * base_HP).

    Solo mosse fisiche/speciali osservate con potenza e accuratezza note,
    tipi noti e statistiche base positive. Una vera immunita vale zero;
    un insieme di mosse non valutabile restituisce None.
    Non usa boost, status, PP, priorita, meteo o statistiche effettive.
    La salute dell'attaccante non riduce la potenza del suo singolo attacco.
    Il denominatore base_HP * HP% e un proxy, non gli HP effettivi del gioco.
    """
    metadata = context['metadata']
    attacking_data = metadata.get(attacker_name, {})
    defending_data = metadata.get(defender_name, {})
    attacker_types, defender_types = types_of(attacking_data), types_of(defending_data)
    hp_base = number(defending_data.get('base_hp'))
    if hp_base is None or hp_base <= 0 or not attacker_types or not defender_types:
        return None
    if any(t not in KNOWN_TYPES for t in (*attacker_types, *defender_types)):
        return None
    scores = []
    for move in attacker['moves'].values():
        stats = {'physical': ('base_atk', 'base_def'),
                 'special': ('base_spa', 'base_spd')}.get(move['category'])
        power, accuracy = number(move['power']), number(move['accuracy'])
        if stats is None or power is None or power <= 0 or accuracy is None or not 0 <= accuracy <= 1:
            continue
        attack, defense = number(attacking_data.get(stats[0])), number(defending_data.get(stats[1]))
        effectiveness = type_effectiveness(move['type'], defender_types)
        if attack is None or attack <= 0 or defense is None or defense <= 0 or effectiveness is None:
            continue
        stab = 1.5 if move['type'] in attacker_types else 1.0
        scores.append(power * accuracy * stab * effectiveness * attack / defense)
    return max(scores) / (defender['hp'] * hp_base) if scores else None


def _potential_matrices(context, alive):
    if not TYPE_MULTIPLIER:
        raise ValueError('Configurare type_chart prima dei nuovi indici di confronto.')
    matrices = {}
    for side, opponent in (('p1', 'p2'), ('p2', 'p1')):
        matrix = np.full((len(alive[side]), len(alive[opponent])), np.nan)
        for i, (name, state) in enumerate(alive[side]):
            for j, (other_name, other_state) in enumerate(alive[opponent]):
                value = _potential_pair_pressure(context, name, state, other_name, other_state)
                if value is not None:
                    matrix[i, j] = value
        matrices[side] = matrix
    return matrices


## 2. Feature finali: stato, mosse, duello, andamento e grafo


In [3]:
def _base50_keys():
    keys = []
    for side in ("p1", "p2"):
        keys += [f"{side}_{metric}" for metric in (
            "remaining", "alive_observed_hp_sum", "alive_hp_unknown_count",
            "alive_under25", "alive_status_slp", "alive_status_par",
            "alive_status_brn", "alive_status_poison",
        )]
        keys += [f"active_{side}_{metric}" for metric in (
            "hp", "hp_known", "boost_atk", "boost_spa", "boost_spe",
            "hp_x_boost_atk", "hp_x_boost_spa",
        )]
        keys += [f"timeline_{side}_21_30_{metric}" for metric in (
            "hp_loss", "hp_recovery", "new_ko",
        )]
        keys += [f"occupancy_{side}_all_{metric}" for metric in (
            "active_hp_mean", "recorded_base_power_per_turn", "bad_status_fraction",
        )]
        keys += [f"coverage_{side}_{metric}" for metric in (
            "offensive_pressure", "worst_response",
            "attackers_evaluable_fraction", "defenders_evaluable_fraction",
        )]
    return tuple(keys)


BASE50_KEYS = _base50_keys()
MOVE_PLACEHOLDERS = {
    "", "none", "missing", "unknown", "null", "notype",
    "no_move", "nomove", "noaction", "no_action", "switch",
}


def extract_base50_from_context(context):
    features = timeline_features(context)
    resource_features(features, context)
    active_features(features, context)
    features.update(occupancy_features(context))
    features.update(offensive_coverage_features(context))

    for side in SIDES:
        features[f"{side}_alive_status_poison"] = (
            features[f"{side}_alive_status_psn"]
            + features[f"{side}_alive_status_tox"]
        )
        features[f"occupancy_{side}_all_bad_status_fraction"] = sum(
            features[f"occupancy_{side}_all_status_{status}_fraction"]
            for status in STATUSES[1:]
        )
        for boost in ("atk", "spa"):
            features[f"active_{side}_hp_x_boost_{boost}"] = (
                features[f"active_{side}_hp"]
                * features[f"active_{side}_boost_{boost}"]
            )

    result = {key: float(features[key]) for key in BASE50_KEYS}
    if len(result) != 50:
        raise ValueError(f"Attese 50 feature base, trovate {len(result)}.")
    return result


def normalized_move_name(value):
    value = str(value).strip().lower().replace(" ", "_") if value is not None else ""
    return "" if value in MOVE_PLACEHOLDERS else value


def extract_move_count_features(battle):
    counts = {"p1": Counter(), "p2": Counter()}
    for event in observed_timeline(battle):
        for side in SIDES:
            move_name = normalized_move_name(
                (event.get(f"{side}_move_details") or {}).get("name")
            )
            if move_name:
                counts[side][move_name] += 1

    return {
        f"observed_move_{side}={move_name}": float(count)
        for side in SIDES
        for move_name, count in counts[side].items()
    }


FINAL_STATUSES = [
    "nostatus", "slp", "par", "tox",
    "psn", "brn", "frz", "fnt"
]


def _strategy_stage_multiplier(stage):
    """Moltiplicatore delle statistiche per gli stage di Gen 2."""
    stage = float(stage or 0)
    stage = max(-6.0, min(6.0, stage))

    if stage >= 0:
        return (2.0 + stage) / 2.0

    return 2.0 / (2.0 - stage)


def _strategy_effective_speed(context, side, name, state):
    """
    Velocità stimata.
    I boost vengono considerati soltanto se il Pokémon è quello attivo,
    perché cambiando Pokémon i boost vengono persi.
    """
    metadata = context.get("metadata", {}).get(name, {})
    base_speed = number(metadata.get("base_spe"))

    if base_speed is None:
        return np.nan

    speed = float(base_speed)

    if context.get("active", {}).get(side) == name:
        speed *= _strategy_stage_multiplier(
            (state.get("boosts", {}) or {}).get("spe", 0)
        )

    if clean(state.get("status", "nostatus")) == "par":
        speed /= 4.0

    return speed


def _strategy_is_fainted(state):
    hp = number(state.get("hp"))

    return int(
        bool(state.get("ko", False))
        or (hp is not None and hp <= 0)
        or clean(state.get("status", "")) == "fnt"
    )


def _strategy_final_duel_features(context, alive, matrices):
    features = {}

    active_names = {
        side: context.get("active", {}).get(side)
        for side in ["p1", "p2"]
    }

    active_states = {}

    for side in ["p1", "p2"]:
        name = active_names[side]
        active_states[side] = (
            context.get("states", {}).get(side, {}).get(name, {})
            if name is not None
            else {}
        )

    p1_fainted = _strategy_is_fainted(active_states["p1"])
    p2_fainted = _strategy_is_fainted(active_states["p2"])

    features["strategy_final_p1_active_fainted"] = p1_fainted
    features["strategy_final_p2_active_fainted"] = p2_fainted
    features["strategy_final_both_active_fainted"] = int(
        p1_fainted and p2_fainted
    )

    # Positivo se P2 deve sostituire il Pokémon e P1 no.
    features["strategy_final_replacement_advantage_p1"] = (
        p2_fainted - p1_fainted
    )

    speeds = {}

    for side in ["p1", "p2"]:
        name = active_names[side]
        state = active_states[side]
        prefix = f"strategy_final_{side}"

        hp = number(state.get("hp"))
        hp_known = int(hp is not None)
        hp = float(hp) if hp is not None else 0.0

        status = clean(state.get("status", "nostatus"))
        if status not in FINAL_STATUSES:
            status = "nostatus"

        boosts = state.get("boosts", {}) or {}

        features[f"{prefix}_hp"] = hp
        features[f"{prefix}_hp_known"] = hp_known
        features[f"{prefix}_def_boost"] = float(boosts.get("def", 0) or 0) / 6.0
        features[f"{prefix}_spd_boost"] = float(boosts.get("spd", 0) or 0) / 6.0
        features[f"{prefix}_positive_boosts"] = (
            sum(max(float(v or 0), 0.0) for v in boosts.values()) / 18.0
        )

        for possible_status in FINAL_STATUSES:
            features[f"{prefix}_status_{possible_status}"] = int(
                status == possible_status
            )

        speed = _strategy_effective_speed(context, side, name, state)
        speeds[side] = speed

        features[f"{prefix}_effective_speed"] = (
            float(speed) / 255.0 if np.isfinite(speed) else 0.0
        )
        features[f"{prefix}_speed_known"] = int(np.isfinite(speed))

    if np.isfinite(speeds["p1"]) and np.isfinite(speeds["p2"]):
        features["strategy_final_speed_difference"] = (
            speeds["p1"] - speeds["p2"]
        ) / 255.0

        features["strategy_final_p1_faster"] = int(
            speeds["p1"] >= speeds["p2"]
        )
    else:
        features["strategy_final_speed_difference"] = 0.0
        features["strategy_final_p1_faster"] = 0

    alive_names = {
        side: [name for name, _ in alive[side]]
        for side in ["p1", "p2"]
    }

    p1_pressure = np.nan
    p2_pressure = np.nan

    p1_name = active_names["p1"]
    p2_name = active_names["p2"]

    if (
        not p1_fainted
        and not p2_fainted
        and p1_name in alive_names["p1"]
        and p2_name in alive_names["p2"]
    ):
        p1_index = alive_names["p1"].index(p1_name)
        p2_index = alive_names["p2"].index(p2_name)

        p1_pressure = matrices["p1"][p1_index, p2_index]
        p2_pressure = matrices["p2"][p2_index, p1_index]

    features["strategy_final_p1_pressure_known"] = int(
        np.isfinite(p1_pressure)
    )
    features["strategy_final_p2_pressure_known"] = int(
        np.isfinite(p2_pressure)
    )

    features["strategy_final_p1_pressure"] = (
        np.log1p(max(float(p1_pressure), 0.0))
        if np.isfinite(p1_pressure)
        else 0.0
    )

    features["strategy_final_p2_pressure"] = (
        np.log1p(max(float(p2_pressure), 0.0))
        if np.isfinite(p2_pressure)
        else 0.0
    )

    if np.isfinite(p1_pressure) and np.isfinite(p2_pressure):
        features["strategy_final_pressure_difference"] = (
            np.log1p(max(float(p1_pressure), 0.0))
            - np.log1p(max(float(p2_pressure), 0.0))
        )

        features["strategy_final_p1_favorable"] = int(
            p1_pressure >= p2_pressure
        )
    else:
        features["strategy_final_pressure_difference"] = 0.0
        features["strategy_final_p1_favorable"] = 0

    features["strategy_final_p1_fast_and_favorable"] = int(
        features["strategy_final_p1_faster"] == 1
        and features["strategy_final_p1_favorable"] == 1
    )

    return features


def extract_final_features(battle):
    context = collect_battle(battle)
    alive = {
        side: _potential_alive(context, side)
        for side in SIDES
    }
    matrices = _potential_matrices(context, alive)

    features = extract_base50_from_context(context)
    features.update(extract_move_count_features(battle))
    features.update(_strategy_final_duel_features(context, alive, matrices))

    if not np.isfinite(list(features.values())).all():
        raise ValueError(f"Feature non finite nella battle {battle.get('battle_id')}.")
    return features


In [4]:
# Blocchi aggiuntivi compatti. Nessuna regola ibrida: l'affidabilità del grafo
# diventa una feature che il singolo XGBoost può usare.

TEMPORAL_WINDOWS_5 = tuple(
    (start, start + 4)
    for start in range(1, 31, 5)
)
BAD_FINAL_STATUSES = {"brn", "frz", "par", "psn", "slp", "tox"}


def _mean_or_zero(values):
    values = [
        float(value)
        for value in values
        if value is not None and np.isfinite(value)
    ]
    return float(np.mean(values)) if values else 0.0


def _snapshot_at_turn(context, stop_turn):
    snapshot = {}

    for side in SIDES:
        latest = {}

        for row in context["records"][side]:
            if row["turn"] > stop_turn:
                continue

            name = row["active_name"]
            if not name:
                continue

            state = latest.setdefault(
                name,
                {"hp": None, "status": None, "ko": False},
            )

            if row["active_hp"] is not None:
                state["hp"] = float(row["active_hp"])

            if row["active_status"] is not None:
                state["status"] = row["active_status"]

            if row["new_ko"] or state["status"] == "fnt" or state["hp"] == 0:
                state["ko"] = True
                state["hp"] = 0.0
                state["status"] = "fnt"

        alive = [
            state
            for state in latest.values()
            if not state["ko"]
        ]
        known_hp = [
            state["hp"]
            for state in alive
            if state["hp"] is not None
        ]

        snapshot[f"{side}_seen"] = float(len(latest))
        snapshot[f"{side}_fainted"] = float(
            sum(state["ko"] for state in latest.values())
        )
        snapshot[f"{side}_observed_hp"] = float(sum(known_hp))
        snapshot[f"{side}_statused"] = float(
            sum(
                state["status"] in BAD_FINAL_STATUSES
                for state in alive
            )
        )

    return snapshot


def extract_compact_temporal_features(context):
    """Sei finestre da 5 turni più trend; niente identità turno-specifiche."""

    features = {}
    directed = {
        "ko_advantage_p1": [],
        "hp_balance_p1": [],
        "active_hp_difference": [],
        "status_advantage_p1": [],
        "snapshot_hp_difference": [],
        "snapshot_fainted_advantage_p1": [],
    }

    for start, stop in TEMPORAL_WINDOWS_5:
        prefix = f"temporal_{start:02d}_{stop:02d}"
        side_metrics = {}

        for side in SIDES:
            rows = [
                row
                for row in context["records"][side]
                if start <= row["turn"] <= stop
            ]
            consecutive_deltas = [
                row["hp_delta"]
                for row in rows
                if row["hp_delta"] is not None
                and row["hp_gap"] == 1
            ]
            observed_hp = [
                row["active_hp"]
                for row in rows
                if row["active_hp"] is not None
            ]
            known_statuses = [
                row["active_status"]
                for row in rows
                if row["active_status"] is not None
            ]

            side_metrics[side] = {
                "ko": float(sum(row["new_ko"] for row in rows)),
                "switches": float(sum(row["switch"] for row in rows)),
                "hp_loss": float(sum(max(-delta, 0.0) for delta in consecutive_deltas)),
                "hp_recovery": float(sum(max(delta, 0.0) for delta in consecutive_deltas)),
                "active_hp": _mean_or_zero(observed_hp),
                "bad_status_fraction": (
                    sum(status in BAD_FINAL_STATUSES for status in known_statuses)
                    / len(known_statuses)
                    if known_statuses
                    else 0.0
                ),
            }

        snapshot = _snapshot_at_turn(context, stop)

        values = {
            "ko_advantage_p1": (
                side_metrics["p2"]["ko"]
                - side_metrics["p1"]["ko"]
            ),
            "hp_balance_p1": (
                side_metrics["p2"]["hp_loss"]
                - side_metrics["p1"]["hp_loss"]
                + side_metrics["p1"]["hp_recovery"]
                - side_metrics["p2"]["hp_recovery"]
            ),
            "switch_difference": (
                side_metrics["p1"]["switches"]
                - side_metrics["p2"]["switches"]
            ),
            "active_hp_difference": (
                side_metrics["p1"]["active_hp"]
                - side_metrics["p2"]["active_hp"]
            ),
            "status_advantage_p1": (
                side_metrics["p2"]["bad_status_fraction"]
                - side_metrics["p1"]["bad_status_fraction"]
            ),
            "snapshot_hp_difference": (
                snapshot["p1_observed_hp"]
                - snapshot["p2_observed_hp"]
            ),
            "snapshot_fainted_advantage_p1": (
                snapshot["p2_fainted"]
                - snapshot["p1_fainted"]
            ),
            "snapshot_seen_difference": (
                snapshot["p1_seen"]
                - snapshot["p2_seen"]
            ),
        }

        for name, value in values.items():
            features[f"{prefix}_{name}"] = float(value)
            if name in directed:
                directed[name].append(float(value))

    x_axis = np.arange(len(TEMPORAL_WINDOWS_5), dtype=float)

    for name, values in directed.items():
        values = np.asarray(values, dtype=float)
        features[f"temporal_trend_{name}"] = float(
            np.polyfit(x_axis, values, 1)[0]
        )
        features[f"temporal_late_minus_early_{name}"] = float(
            values[-2:].mean() - values[:2].mean()
        )

    return features


def _graph_alive(context, side):
    """P1 mai entrato è sicuramente vivo e intatto; P2 ignoto resta escluso."""

    result = []

    for name, original_state in context["states"][side].items():
        if original_state["ko"]:
            continue

        state = dict(original_state)
        hp = state.get("hp")

        if hp is None:
            if side == "p1" and state.get("seen_turn") is None:
                state["hp"] = 1.0
            else:
                continue

        if state["hp"] > 0:
            result.append((name, state))

    return result


def _member_graph_values(context, name, state):
    metadata = context["metadata"].get(name, {})
    hp = float(state.get("hp") or 0.0)
    base_hp = float(number(metadata.get("base_hp")) or 0.0)
    base_atk = float(number(metadata.get("base_atk")) or 0.0)
    base_def = float(number(metadata.get("base_def")) or 0.0)
    base_spa = float(number(metadata.get("base_spa")) or 0.0)
    base_spd = float(number(metadata.get("base_spd")) or 0.0)
    base_spe = float(number(metadata.get("base_spe")) or 0.0)

    return {
        "hp": hp,
        "healthy": float(hp >= 0.75),
        "offensive": hp * max(base_atk, base_spa),
        "physical_bulk": hp * base_hp * base_def / 100.0,
        "special_bulk": hp * base_hp * base_spd / 100.0,
        "speed": base_spe,
    }


def _coverage_summary(matrix):
    if matrix.size == 0 or matrix.shape[1] == 0:
        return 0.0, 0.0

    best_responses = []

    for column in range(matrix.shape[1]):
        values = matrix[:, column]
        values = values[np.isfinite(values)]

        if len(values):
            best_responses.append(float(np.max(values)))

    coverage_fraction = len(best_responses) / matrix.shape[1]
    mean_pressure = _mean_or_zero(
        np.log1p(np.maximum(best_responses, 0.0))
    )
    return mean_pressure, float(coverage_fraction)


def _conservative_sweeper_score(matrix):
    if matrix.size == 0 or matrix.shape[1] == 0:
        return 0.0

    scores = []

    for row in matrix:
        if np.isfinite(row).all():
            scores.append(
                float(np.min(np.log1p(np.maximum(row, 0.0))))
            )

    return max(scores, default=0.0)


def extract_remaining_graph_features(context, alive, matrices):
    """Riassunti del grafo osservato, con affidabilità esplicita per P2."""

    features = {}
    p2_seen = sum(
        state["seen_turn"] is not None
        for state in context["states"]["p2"].values()
    )
    p2_revealed_fraction = min(1.0, p2_seen / 6.0)

    features["graph_p2_seen"] = float(p2_seen)
    features["graph_p2_unrevealed"] = float(max(0, 6 - p2_seen))
    features["graph_p2_revealed_fraction"] = float(p2_revealed_fraction)
    features["graph_p2_complete"] = float(p2_seen == 6)
    features["graph_p1_unused"] = float(
        sum(
            state["seen_turn"] is None
            for state in context["states"]["p1"].values()
        )
    )

    summary = {}

    for side in SIDES:
        members = [
            _member_graph_values(context, name, state)
            for name, state in alive[side]
        ]

        side_summary = {
            "modeled_alive": float(len(members)),
            "mean_hp": _mean_or_zero(member["hp"] for member in members),
            "healthy_count": float(sum(member["healthy"] for member in members)),
            "mean_offensive": _mean_or_zero(member["offensive"] for member in members),
            "mean_physical_bulk": _mean_or_zero(
                member["physical_bulk"] for member in members
            ),
            "mean_special_bulk": _mean_or_zero(
                member["special_bulk"] for member in members
            ),
            "max_speed": max(
                (member["speed"] for member in members),
                default=0.0,
            ),
        }
        summary[side] = side_summary

        for name, value in side_summary.items():
            features[f"graph_{side}_{name}"] = float(value)

    for name in summary["p1"]:
        difference = summary["p1"][name] - summary["p2"][name]
        features[f"graph_observed_{name}_difference"] = float(difference)

    p1_matrix = matrices["p1"]
    p2_matrix = matrices["p2"]
    aligned_p2 = p2_matrix.T

    pair_total = int(p1_matrix.size)
    valid_pair_mask = (
        np.isfinite(p1_matrix)
        & np.isfinite(aligned_p2)
    ) if p1_matrix.shape == aligned_p2.shape else np.zeros_like(
        p1_matrix,
        dtype=bool,
    )

    features["graph_pair_total"] = float(pair_total)
    features["graph_pair_both_directions_known_fraction"] = (
        float(valid_pair_mask.sum() / pair_total)
        if pair_total
        else 0.0
    )

    if valid_pair_mask.any():
        dominance = (
            np.log1p(np.maximum(p1_matrix[valid_pair_mask], 0.0))
            - np.log1p(np.maximum(aligned_p2[valid_pair_mask], 0.0))
        )
        features["graph_mean_dominance_p1"] = float(np.mean(dominance))
        features["graph_favorable_pair_fraction_p1"] = float(
            np.mean(dominance >= 0)
        )
        features["graph_best_pair_dominance_p1"] = float(np.max(dominance))
        features["graph_worst_pair_dominance_p1"] = float(np.min(dominance))
    else:
        features["graph_mean_dominance_p1"] = 0.0
        features["graph_favorable_pair_fraction_p1"] = 0.0
        features["graph_best_pair_dominance_p1"] = 0.0
        features["graph_worst_pair_dominance_p1"] = 0.0

    p1_coverage, p1_coverage_fraction = _coverage_summary(p1_matrix)
    p2_coverage, p2_coverage_fraction = _coverage_summary(p2_matrix)

    features["graph_p1_coverage_pressure"] = p1_coverage
    features["graph_p2_coverage_pressure"] = p2_coverage
    features["graph_coverage_pressure_difference"] = (
        p1_coverage - p2_coverage
    )
    features["graph_p1_coverage_fraction"] = p1_coverage_fraction
    features["graph_p2_coverage_fraction"] = p2_coverage_fraction

    p1_sweeper = _conservative_sweeper_score(p1_matrix)
    p2_sweeper = _conservative_sweeper_score(p2_matrix)

    features["graph_p1_conservative_sweeper"] = p1_sweeper
    features["graph_p2_conservative_sweeper"] = p2_sweeper
    features["graph_conservative_sweeper_difference"] = (
        p1_sweeper - p2_sweeper
    )

    matchup_reliability = (
        p2_revealed_fraction
        * min(p1_coverage_fraction, p2_coverage_fraction)
    )
    features["graph_matchup_reliability"] = float(matchup_reliability)

    for name in (
        "graph_mean_dominance_p1",
        "graph_coverage_pressure_difference",
        "graph_conservative_sweeper_difference",
        "graph_observed_healthy_count_difference",
        "graph_observed_mean_hp_difference",
    ):
        features[f"reliable_{name}"] = (
            float(features[name]) * matchup_reliability
        )

    return features


# Ridefinizione finale: conserva base 50, mosse e duello e aggiunge
# soltanto i due blocchi compatti sopra.
def extract_final_features(battle):
    context = collect_battle(battle)
    alive = {
        side: _graph_alive(context, side)
        for side in SIDES
    }
    matrices = _potential_matrices(context, alive)

    features = extract_base50_from_context(context)
    features.update(extract_move_count_features(battle))
    features.update(
        _strategy_final_duel_features(
            context,
            alive,
            matrices,
        )
    )
    features.update(
        extract_compact_temporal_features(context)
    )
    features.update(
        extract_remaining_graph_features(
            context,
            alive,
            matrices,
        )
    )

    if not np.isfinite(list(features.values())).all():
        raise ValueError(
            f"Feature non finite nella battle {battle.get('battle_id')}."
        )

    return features



## 3. Estrazione train e test


In [5]:
print("Estrazione feature: stato + mosse + duello + temporali + grafo...")

selected_train_features = [
    extract_final_features(battle)
    for battle in tqdm(train_data, desc="Train features")
]

selected_test_features = [
    extract_final_features(battle)
    for battle in tqdm(test_data, desc="Test features")
]

y_array = np.asarray(
    [int(battle["player_won"]) for battle in train_data],
    dtype=np.int8,
)

if len(selected_train_features) != len(y_array):
    raise ValueError("Feature train e target non allineati.")
if len(selected_test_features) != len(test_data):
    raise ValueError("Feature test e test_data non allineati.")
if set(np.unique(y_array)) != {0, 1}:
    raise ValueError("Il target deve contenere entrambe le classi 0 e 1.")

fixed_prefixes = {
    "duello": "strategy_final_",
    "temporali": "temporal_",
    "grafo": "graph_",
    "grafo_affidabile": "reliable_graph_",
}

schema_train = {
    name: {
        key
        for row in selected_train_features
        for key in row
        if key.startswith(prefix)
    }
    for name, prefix in fixed_prefixes.items()
}
schema_test = {
    name: {
        key
        for row in selected_test_features
        for key in row
        if key.startswith(prefix)
    }
    for name, prefix in fixed_prefixes.items()
}

if schema_train != schema_test:
    raise ValueError(
        "Schema delle feature fisse non coerente tra train e test."
    )

family_rows = [
    {
        "famiglia": name,
        "numero_feature": len(schema_train[name]),
    }
    for name in fixed_prefixes
]
family_rows.append({
    "famiglia": "mosse sparse osservate nel train",
    "numero_feature": len({
        key
        for row in selected_train_features
        for key in row
        if key.startswith("observed_move_")
    }),
})

display(pd.DataFrame(family_rows))
print("Righe train:", len(selected_train_features))
print("Righe test:", len(selected_test_features))


Estrazione feature: stato + mosse + duello + temporali + grafo...


Train features:   0%|          | 0/10000 [00:00<?, ?it/s]

Test features:   0%|          | 0/5000 [00:00<?, ?it/s]

,famiglia,numero_feature
0,duello,43
1,temporali,60
2,grafo,41
3,grafo_affidabile,5
4,mosse sparse osservate nel train,432


Righe train: 10000
Righe test: 5000


## 4. Ablation, tuning e validation indipendente

Il 20% del train viene isolato subito. Sul restante 80% confrontiamo quattro blocchi di feature con la stessa cross-validation, scegliamo il migliore e facciamo il tuning soltanto su quello. L'holdout viene usato una sola volta per stimare l'accuracy del singolo XGBoost finale.


In [6]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted
from xgboost import XGBClassifier


class BattleVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, min_generic=5, min_moves=20):
        self.min_generic = min_generic
        self.min_moves = min_moves

    def fit(self, X, y=None):
        self.vectorizer_ = DictVectorizer(
            sparse=True,
            dtype=np.float32,
        )
        matrix = self.vectorizer_.fit_transform(X)
        names = self.vectorizer_.get_feature_names_out()
        counts = np.asarray(
            (matrix != 0).sum(axis=0)
        ).ravel()
        thresholds = np.asarray([
            (
                self.min_moves
                if name.startswith("observed_move_")
                else self.min_generic
            )
            for name in names
        ])
        self.mask_ = counts >= thresholds

        if not self.mask_.any():
            raise ValueError(
                "Nessuna feature supera il filtro di frequenza."
            )

        self.feature_names_ = names[self.mask_]
        return self

    def transform(self, X):
        check_is_fitted(
            self,
            ["vectorizer_", "mask_", "feature_names_"],
        )
        matrix = self.vectorizer_.transform(X)[:, self.mask_]

        if not np.isfinite(matrix.data).all():
            raise ValueError(
                "La matrice contiene valori non finiti."
            )

        return matrix

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "feature_names_")
        return self.feature_names_.copy()


OPTIONAL_FEATURE_PREFIXES = (
    "temporal_",
    "graph_",
    "reliable_graph_",
)

FEATURE_SETS = {
    "Base + mosse + duello": (),
    "Base + temporali": (
        "temporal_",
    ),
    "Base + grafo": (
        "graph_",
        "reliable_graph_",
    ),
    "Tutte": OPTIONAL_FEATURE_PREFIXES,
}


def filter_feature_rows(rows, enabled_prefixes):
    enabled_prefixes = tuple(enabled_prefixes)

    return [
        {
            key: value
            for key, value in row.items()
            if (
                not key.startswith(
                    OPTIONAL_FEATURE_PREFIXES
                )
                or key.startswith(
                    enabled_prefixes
                )
            )
        }
        for row in rows
    ]


all_indices = np.arange(len(y_array))

# Nuovo holdout: non riutilizziamo quello gi? osservato nel run precedente.
build_indices, holdout_indices = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=314159,
    stratify=y_array,
)

y_build = y_array[build_indices]
y_holdout = y_array[holdout_indices]

inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


def make_reference_pipeline():
    return Pipeline([
        (
            "features",
            BattleVectorizer(
                min_generic=5,
                min_moves=20,
            ),
        ),
        (
            "model",
            XGBClassifier(
                n_estimators=500,
                learning_rate=0.03,
                max_depth=4,
                min_child_weight=5,
                subsample=0.85,
                colsample_bytree=0.75,
                reg_alpha=0.10,
                reg_lambda=5.0,
                gamma=0.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=1,
                **XGB_GPU_PARAMS,
            ),
        ),
    ])


ablation_scores = {}
ablation_rows = []

print(
    "Ablation dei blocchi sul solo build set:",
    f"{len(build_indices):,} battaglie",
    flush=True,
)

for feature_set_name, enabled_prefixes in FEATURE_SETS.items():
    filtered_rows = filter_feature_rows(
        selected_train_features,
        enabled_prefixes,
    )
    X_build_candidate = [
        filtered_rows[index]
        for index in build_indices
    ]

    scores = cross_val_score(
        make_reference_pipeline(),
        X_build_candidate,
        y_build,
        scoring="accuracy",
        cv=inner_cv,
        n_jobs=1,
    )
    ablation_scores[feature_set_name] = scores

    raw_feature_count = len({
        key
        for row in X_build_candidate
        for key in row
    })

    ablation_rows.append({
        "feature_set": feature_set_name,
        "accuracy_cv_media": scores.mean(),
        "deviazione_standard": scores.std(),
        "accuracy_minima": scores.min(),
        "accuracy_massima": scores.max(),
        "numero_feature_raw": raw_feature_count,
    })

ablation_results = (
    pd.DataFrame(ablation_rows)
    .sort_values(
        "accuracy_cv_media",
        ascending=False,
        kind="stable",
    )
    .reset_index(drop=True)
)

display(
    ablation_results.style.format({
        "accuracy_cv_media": "{:.4f}",
        "deviazione_standard": "{:.4f}",
        "accuracy_minima": "{:.4f}",
        "accuracy_massima": "{:.4f}",
    })
)

selected_feature_set_name = str(
    ablation_results.loc[0, "feature_set"]
)
selected_prefixes = FEATURE_SETS[
    selected_feature_set_name
]

print(
    "Blocco scelto senza usare l'holdout:",
    selected_feature_set_name,
)

model_train_features = filter_feature_rows(
    selected_train_features,
    selected_prefixes,
)

model_test_features = filter_feature_rows(
    selected_test_features,
    selected_prefixes,
)

X_build_selected = [
    model_train_features[index]
    for index in build_indices
]
X_holdout_selected = [
    model_train_features[index]
    for index in holdout_indices
]

xgb_pipeline = make_reference_pipeline()

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions={
        "features__min_generic": [3, 5, 10],
        "features__min_moves": [10, 20, 30],
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [3, 4, 5],
        "model__learning_rate": [
            0.02,
            0.03,
            0.05,
            0.07,
        ],
        "model__min_child_weight": [
            3,
            5,
            10,
            20,
        ],
        "model__subsample": [0.75, 0.85, 1.0],
        "model__colsample_bytree": [
            0.70,
            0.85,
            1.0,
        ],
        "model__reg_alpha": [0.0, 0.1, 1.0],
        "model__reg_lambda": [
            3.0,
            5.0,
            10.0,
            20.0,
        ],
        "model__gamma": [0.0, 0.2, 1.0],
    },
    n_iter=20,
    scoring="accuracy",
    cv=inner_cv,
    n_jobs=1,
    pre_dispatch=1,
    random_state=42,
    refit=True,
    verbose=1,
    error_score="raise",
)

print(
    "Tuning del blocco scelto:",
    selected_feature_set_name,
    flush=True,
)
xgb_search.fit(
    X_build_selected,
    y_build,
)

holdout_probabilities = (
    xgb_search.best_estimator_
    .predict_proba(
        X_holdout_selected
    )[:, 1]
)
holdout_predictions = (
    holdout_probabilities >= 0.5
).astype(np.int8)
holdout_accuracy = accuracy_score(
    y_holdout,
    holdout_predictions,
)

best_search = xgb_search
best_model_name = "XGBoost singolo"

comparison = pd.DataFrame([{
    "modello": best_model_name,
    "feature_set": selected_feature_set_name,
    "accuracy_cv_tuning": xgb_search.best_score_,
    "std_cv": xgb_search.cv_results_[
        "std_test_score"
    ][xgb_search.best_index_],
    "accuracy_holdout_indipendente": holdout_accuracy,
    "numero_feature_finali": len(
        xgb_search.best_estimator_[
            "features"
        ].get_feature_names_out()
    ),
}])
display(comparison)

display(pd.DataFrame(
    confusion_matrix(
        y_holdout,
        holdout_predictions,
    ),
    index=["Reale 0", "Reale 1"],
    columns=["Predetto 0", "Predetto 1"],
))
print(classification_report(
    y_holdout,
    holdout_predictions,
    digits=4,
))

print("Parametri selezionati:")
display(pd.Series(
    xgb_search.best_params_,
    name="valore",
).to_frame())

feature_names = (
    xgb_search.best_estimator_["features"]
    .get_feature_names_out()
)
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": (
        xgb_search.best_estimator_["model"]
        .feature_importances_
    ),
}).sort_values(
    "importance",
    ascending=False,
)

print("Feature pi? utilizzate sul build set:")
display(feature_importance.head(30))


Ablation dei blocchi sul solo build set: 8,000 battaglie


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [15:50:22] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,feature_set,accuracy_cv_media,deviazione_standard,accuracy_minima,accuracy_massima,numero_feature_raw
0,Base + mosse + duello,0.6774,0.0052,0.6713,0.6850,510
1,Base + grafo,0.6743,0.0056,0.6656,0.6806,556
2,Tutte,0.6727,0.0091,0.6613,0.6887,616
3,Base + temporali,0.6696,0.0075,0.6619,0.6837,570


Blocco scelto senza usare l'holdout: Base + mosse + duello
Tuning del blocco scelto: Base + mosse + duello
Fitting 5 folds for each of 20 candidates, totalling 100 fits


,modello,feature_set,accuracy_cv_tuning,std_cv,accuracy_holdout_indipendente,numero_feature_finali
0,XGBoost singolo,Base + mosse + duello,0.675125,0.00522,0.689,296


,Predetto 0,Predetto 1
Reale 0,652,348
Reale 1,274,726


              precision    recall  f1-score   support

           0     0.7041    0.6520    0.6771      1000
           1     0.6760    0.7260    0.7001      1000

    accuracy                         0.6890      2000
   macro avg     0.6900    0.6890    0.6886      2000
weighted avg     0.6900    0.6890    0.6886      2000

Parametri selezionati:


,valore
model__subsample,0.75
model__reg_lambda,20.00
model__reg_alpha,1.00
model__n_estimators,500.00
model__min_child_weight,20.00
model__max_depth,5.00
model__learning_rate,0.03
model__gamma,1.00
model__colsample_bytree,1.00
features__min_moves,30.00


Feature pi? utilizzate sul build set:


,feature,importance
245,p2_alive_under25,0.026177
238,p1_remaining,0.025592
288,strategy_final_replacement_advantage_p1,0.022834
239,p2_alive_hp_unknown_count,0.013155
246,p2_remaining,0.011846
292,timeline_p1_21_30_new_ko,0.011345
15,coverage_p1_defenders_evaluable_fraction,0.011318
232,p1_alive_observed_hp_sum,0.010573
295,timeline_p2_21_30_new_ko,0.010443
240,p2_alive_observed_hp_sum,0.009856


## 5. Modello finale e submission


In [8]:
# Il modello selezionato era stato addestrato soltanto sul build set.
# Ora lo cloniamo e lo riaddestriamo su tutte le 10.000 battaglie,
# usando lo stesso feature set impiegato per validation e test.
final_model = clone(
    best_search.best_estimator_
)

final_model.fit(
    model_train_features,
    y_array,
)

test_predictions = np.asarray(
    final_model.predict(model_test_features),
    dtype=int,
)

if test_predictions.shape != (len(test_data),):
    raise ValueError(
        "Numero di predizioni test non corretto."
    )

if not np.isin(test_predictions, [0, 1]).all():
    raise ValueError(
        "Le predizioni devono essere 0 oppure 1."
    )

# Evita di creare nuovamente una submission tutta 0 o tutta 1.
if np.unique(test_predictions).size < 2:
    raise ValueError(
        "Predizioni costanti: controllare che train e test "
        "usino lo stesso feature set."
    )

predictions = pd.DataFrame({
    "battle_id": [
        battle["battle_id"]
        for battle in test_data
    ],
    "player_won": test_predictions,
})

sample_path = DATA_DIR / "sample_submission.csv"

if sample_path.exists():
    sample = pd.read_csv(sample_path)

    submission = sample[["battle_id"]].merge(
        predictions,
        on="battle_id",
        how="left",
        validate="one_to_one",
    )
else:
    submission = predictions.copy()

if submission["player_won"].isna().any():
    raise ValueError(
        "Mancano predizioni per alcuni battle_id."
    )

if submission["battle_id"].duplicated().any():
    raise ValueError(
        "battle_id duplicati nella submission."
    )

submission["player_won"] = (
    submission["player_won"].astype(int)
)

output_path = Path(
    "/kaggle/working/submission_single_xgb_gpu_CORRETTA.csv"
)

submission.to_csv(
    output_path,
    index=False,
)

display(submission.head())

print("Creato:", output_path.resolve())
print("Dimensione:", submission.shape)
print("Modello:", best_model_name)
print("Feature set:", selected_feature_set_name)
print("Accuracy holdout:", round(holdout_accuracy, 4))
print(
    "Distribuzione:",
    submission["player_won"]
    .value_counts()
    .sort_index()
    .to_dict(),
)

,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,0
4,4,1


Creato: /kaggle/working/submission_single_xgb_gpu_CORRETTA.csv
Dimensione: (5000, 2)
Modello: XGBoost singolo
Feature set: Base + mosse + duello
Accuracy holdout: 0.689
Distribuzione: {0: 2372, 1: 2628}
